In [1]:
from utils import load_env_vars
load_env_vars()

from model_mbert2 import MBertWritingReward
import json, random, tqdm
from llms import generate

reward_model = MBertWritingReward("models/mbert2-large-PR-final")

with open("data/annotation_instructions_creative_writing.json", "r") as f:
    instruction_data = json.load(f)

draft_prompt = """Write a [[N_WORDS]] word paragraph in [[VOICE]] based on the content below. Try your best to be original, avoiding clichés, awkward words or overused tropes. Do not use ornamental language and focus on nuance, simplicity, and subtext.

Writing Instruction:
[[INSTRUCTION]]"""

/home/tingotower/anaconda3/lib/python3.9/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")
You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


In [ ]:
import concurrent.futures, tqdm

prompt_fix_typos = """You are given a paragraph of writing. There might be typos, formatting or grammatical errors.
Your task is to rewrite the paragraph fixing such typos, formatting or grammatical errors. Do not change anything else in the paragraph, do not change any of the meaning.
Your response should only be the rewritten paragraph, do not add any other text including "Here is the rewritten paragraph:" or anything like that.

Paragraph:
[[PARAGRAPH]]"""


def generate_cot_candidate(ai_draft):
    cot_model_full_name, cot_model_short_name = "ft:gpt-4o-2024-08-06:salesforce-research:lamp-4o-cot:Aqlv1wPq", "lamp-4o-cot"
    with open("prompts/lamp_cot_input.txt", "r") as f:
        lamp_cot_input_prompt = f.read()
    cot_response = generate([{"role": "user", "content": lamp_cot_input_prompt}], model=cot_model_full_name, variables={"INPUT_PARAGRAPH": ai_draft}, max_tokens=2000, step="lamp-editing-cot")
    part_3_idx = cot_response.find("Part 3")
    final_rewrite = cot_response[part_3_idx:]
    final_rewrite = "\n".join(final_rewrite.split("\n")[1:]).strip()

    final_rewrite_fixed = generate([{"role": "user", "content": prompt_fix_typos}], model="gpt-4o", max_tokens=1000, variables={"PARAGRAPH": final_rewrite})

    return final_rewrite_fixed

def process_instruction_into_triplets(instruction, n_cot_candidates=20, n_workers=5):
    
    first_draft = generate([{"role": "user", "content": draft_prompt}], model="gpt-4o", max_tokens=1000, variables={"INSTRUCTION": instruction['plot'], "N_WORDS": str(instruction['word_count']), "VOICE": instruction['voice']})
    first_draft_score = reward_model.predict_regression(first_draft)
    
    generated_candidates = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=n_workers) as executor:
        futures = [executor.submit(generate_cot_candidate, first_draft) for _ in range(n_cot_candidates)]
        
        for future in tqdm.tqdm(concurrent.futures.as_completed(futures), total=n_cot_candidates, desc="Generating COT candidates"):
            generated_candidates.append(future.result())
    
    first_draft_len = len(first_draft)

    cot_candidates = []
    for candidate in generated_candidates:
        if len(candidate) < 0.6 * first_draft_len:
            continue # it shouldn't be so short
        score = reward_model.predict_regression(candidate)
        cot_candidates.append({"candidate": candidate, "score": score})

    first_draft_candidate = {"id": "first_draft", "candidate": first_draft, "score": first_draft_score}

    cot_candidates = sorted(cot_candidates, key=lambda x: x["score"], reverse=True)

    random_cot_candidate = cot_candidates[len(cot_candidates) // 2]
    random_cot_candidate["id"] = "cot_random"
    best_cot_candidate = cot_candidates[0]
    best_cot_candidate["id"] = "cot_w_ttc"
    return {"instruction": instruction, "first_draft": first_draft_candidate, "random_cot": random_cot_candidate, "best_cot": best_cot_candidate}

dataset_fn = "data/preference_annotation_triplets.jsonl"

dataset = []
for idx, instruction in enumerate(tqdm.tqdm_notebook(instruction_data, desc="Processing instructions")):
    triplets = process_instruction_into_triplets(instruction, n_workers=10)
    triplets["id"] = f"triplet_{idx}"
    # dataset.append(triplets)
    with open(dataset_fn, "a") as f:
        f.write(json.dumps(triplets) + "\n")


/tmp/ipykernel_4081852/2792618541.py:58: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for idx, instruction in enumerate(tqdm.tqdm_notebook(instruction_data, desc="Processing instructions")):


Processing instructions:   0%|          | 0/50 [00:00<?, ?it/s]

Generating COT candidates: 100%|██████████| 20/20 [01:08<00:00,  3.42s/it]
